# Harmonize — Chile climate-finance inventory (OEF)

Harmonize the four Chile climate-finance source reviews into one inventory.
Inputs (authoritative per-source reviews):
  ../../cl-mma/cl-mma-fondos/releases/v1/data/cl_mma_fondos_v1.csv
  ../../cl-minenergia/cl-minenergia-fondos/releases/v1/data/cl_minenergia_programs_v1.csv
  ../../cl-subdere/cl-subdere-fondos/releases/v1/data/cl_subdere_programs_v1.csv
  ../../cl-minvu/cl-minvu-fondos/releases/v1/data/cl_minvu_programs_v1.csv
Output: chile_finance_inventory.csv  (one row per fund/programme line)
This is an OEF exploratory product — NOT a Mage pipeline. Re-run after any source release.

In [2]:
import pandas as pd, json, os
HERE=os.getcwd()  # notebook runs in its own dir
SRC={
 "cl-mma":      "../../../../cl-mma/cl-mma-fondos/releases/v1/data/cl_mma_fondos_v1.csv",
 "cl-minenergia":"../../../../cl-minenergia/cl-minenergia-fondos/releases/v1/data/cl_minenergia_programs_v1.csv",
 "cl-subdere":  "../../../../cl-subdere/cl-subdere-fondos/releases/v1/data/cl_subdere_programs_v1.csv",
 "cl-minvu":    "../../../../cl-minvu/cl-minvu-fondos/releases/v1/data/cl_minvu_programs_v1.csv",
 "cl-gore":     "../../../../cl-gore/cl-gore-fndr/releases/v1/data/cl_gore_fndr_programs_v1.csv",
 "cl-corfo":    "../../../../cl-corfo/cl-corfo-finance/releases/v1/data/cl_corfo_programs_v1.csv",
}
SCHEMA=["source_dataset","funder_institution","program_name","program_family","eligible_actor",
 "instrument_type","amount_clp","amount_note","open_date","close_date","status","recurrence",
 "specificity","climate_relevance","climate_relevance_norm","gpc_sectors","access_pathway",
 "detail_level","status_as_of","source_url","notes"]

def norm_relevance(v):
    s=str(v).lower()
    if s.startswith("explicit-adjacent"): return "adjacent"
    if s.startswith("explicit"): return "explicit"
    if "adjacent" in s: return "adjacent"
    if s.startswith("indirect") or "indirect" in s: return "indirect"
    return "unknown"

def load(ds, path):
    df=pd.read_csv(os.path.join(HERE,path))
    out=pd.DataFrame(index=df.index)
    out["source_dataset"]=ds
    if ds=="cl-mma":
        out["funder_institution"]="Ministerio del Medio Ambiente (MMA)"
        out["program_name"]=df["fund_name"]
        out["program_family"]=df["stream"]
        out["amount_clp"]=df["amount_clp"]
        out["amount_note"]=df["amount_suspect"].map(lambda x:"amount_suspect (source typo)" if x else "")
        out["open_date"]=df["open_date"]; out["close_date"]=df["close_date"]
        out["notes"]=df["fund_program"].astype(str)
    else:
        out["funder_institution"]=df["funder_institution"]
        out["program_name"]=df["program"]
        out["program_family"]=df["program"]
        out["amount_clp"]=pd.NA
        out["amount_note"]=df["amount_note"]
        out["open_date"]=pd.NA; out["close_date"]=pd.NA
        out["notes"]=df["notes"]
    for c in ["eligible_actor","instrument_type","status","recurrence","specificity",
              "climate_relevance","gpc_sectors","access_pathway","detail_level","status_as_of","source_url"]:
        out[c]=df[c]
    out["climate_relevance_norm"]=out["climate_relevance"].map(norm_relevance)
    return out[SCHEMA]

frames=[load(ds,p) for ds,p in SRC.items()]
inv=pd.concat(frames,ignore_index=True)

## Validate & export

In [3]:
# validate
counts=inv.source_dataset.value_counts().to_dict()
assert counts=={"cl-mma":55,"cl-minenergia":6,"cl-subdere":4,"cl-minvu":4,"cl-gore":4,"cl-corfo":5}, counts
assert len(inv)==78, len(inv)
assert inv.source_url.str.startswith("http").all()
assert set(inv.climate_relevance_norm)<= {"explicit","adjacent","indirect","unknown"}
assert inv.program_name.notna().all()
inv.to_csv(os.path.join(HERE,"data/chile_finance_inventory.csv"),index=False)